# Pipette Well Detection — Kaggle GPU Runner

Runs the v11+ training pipeline on a Kaggle T4/P100 GPU. Each cell is
idempotent so the notebook can be safely re-run after a session timeout.

## One-time setup before running the notebook

1. **Upload the dataset.** Create a private Kaggle Dataset containing
   `data/pipette_well_dataset/` (real + synthetic .mp4 files +
   `labels.json` + `labels_synthetic.json`). Suggested slug:
   `pipette-well-dataset`. Attach it to this notebook via
   *Add data → Your Datasets*.

2. **Optional: upload a resume checkpoint.** If you want to resume from
   an existing `best.pt`, create a second Kaggle Dataset containing only
   that file (suggested slug `pipette-well-checkpoint`) and attach it.
   Skip this for a fresh run from DINOv2 pretrained weights.

3. **Enable Internet + GPU.** Settings → enable Internet (needed for
   `git clone` and the DINOv2 weight download) and select GPU T4 x2 (do NOT use P100 — it has compute capability sm_60 which the default Kaggle PyTorch image does not support).

## What this notebook does

1. Print GPU + library info
2. Clone the repo at the configured commit
3. Install dependencies
4. Locate the attached dataset(s)
5. **Smoke tests** — fail fast if anything is wrong:
   - Import + CUDA available
   - `pytest tests/test_leak_free_split.py` (11 split-correctness tests)
   - Model construction + 1-batch forward pass on dummy CUDA tensors
   - Dataset loads a real sample
   - (If resuming) checkpoint loads cleanly into the model
6. Launch training via `run_training.sh` with `DEVICE=cuda`
7. Persist `best.pt` to `/kaggle/working/` so it survives session teardown

## 0. Configuration

Edit these once at the top. Everything below reads from them.

In [ ]:
REPO_URL = 'https://github.com/linwoes/pipette-well-challenge.git'
REPO_REF = 'main'                  # branch, tag, or commit hash
REPO_DIR = '/kaggle/working/pipette-well-challenge'

# Auto-detected if not set; override only if your Kaggle dataset slug is unusual.
DATA_DATASET_SLUG       = None     # e.g. 'pipette-well-dataset'
CHECKPOINT_DATASET_SLUG = None     # e.g. 'pipette-well-checkpoint' (None = fresh start)

# Training knobs (overrides for run_training.sh)
EPOCHS         = 80
BATCH_SIZE     = 2                 # T4 (~14.6 GB) OOMs at 8 once backward-pass activations land; matches run_training.sh default
USE_SYNTHETIC  = 0                 # 0 = real-only (80 train / 20 val) to diagnose synthetic->real gap
WEIGHT_DECAY   = '1e-2'            # 10x increase (was 1e-3) to combat overfitting
LORA_RANK      = 2                 # reduced from 4 to cut trainable params and overfitting
DROPOUT        = 0.5               # fusion MLP + temporal attention dropout (was 0.3)
NUM_FOLDS      = 5                 # K-fold CV: number of folds
FOLD_INDEX     = 0                 # K-fold CV: which fold to use as validation (0-indexed)
TRAINING_VERS  = None              # None -> auto-compute YYYYMMDD.<git-hash>

## 1. Environment info

In [ ]:
import sys, platform, subprocess

print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())

try:
    import torch
    print('Torch    :', torch.__version__)
    print('CUDA     :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device   :', torch.cuda.get_device_name(0))
        print('VRAM     :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('Torch not installed yet (will be after pip install)')

subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'])

## 2. Clone the repo at the configured ref

Idempotent: skips the clone if the directory already exists.

In [ ]:
import os, subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_REF], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'Repo at {REPO_DIR}; HEAD = {head}')

## 3. Install dependencies

Kaggle's base image already has torch + numpy + most heavy deps. We only
need the project-specific packages from `requirements.txt` plus pytest
for the smoke tests.

In [ ]:
!pip install -q -r requirements.txt pytest

## 4. Locate attached datasets

Kaggle mounts datasets at `/kaggle/input/<slug>/`. We auto-detect the
video dataset by looking for `labels.json`, and the (optional) checkpoint
dataset by looking for `best.pt`.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

def find_data_dir() -> Path:
    """Locate a directory containing labels.json + at least one .mp4."""
    if DATA_DATASET_SLUG:
        candidates = [INPUT_ROOT / DATA_DATASET_SLUG]
    else:
        candidates = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for c in candidates:
        for labels in c.rglob('labels.json'):
            if list(labels.parent.glob('*.mp4')):
                return labels.parent
    raise FileNotFoundError(
        f'Could not find a directory with labels.json + *.mp4 under {INPUT_ROOT}. '
        'Did you attach the dataset?'
    )

def find_checkpoint() -> Path | None:
    """Locate best.pt if a checkpoint dataset is attached."""
    if CHECKPOINT_DATASET_SLUG:
        candidates = [INPUT_ROOT / CHECKPOINT_DATASET_SLUG]
    else:
        candidates = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for c in candidates:
        for ckpt in c.rglob('best.pt'):
            return ckpt
    return None

DATA_DIR = find_data_dir()
RESUME_FROM = find_checkpoint()

print(f'DATA_DIR    : {DATA_DIR}')
print(f'  labels.json           : {(DATA_DIR / "labels.json").exists()}')
print(f'  labels_synthetic.json : {(DATA_DIR / "labels_synthetic.json").exists()}')
print(f'  *.mp4 count           : {len(list(DATA_DIR.glob("*.mp4")))}')
print(f'RESUME_FROM : {RESUME_FROM if RESUME_FROM else "(none — will train from DINOv2 pretrained)"}')

## 5. Smoke tests

Five independent checks that catch the common ways a remote run can
silently fail. Each is small and fast (under a minute total). If any
fails, **stop here** rather than burning GPU hours on a broken run.

### Smoke test 1: imports + CUDA available

In [ ]:
import torch
from src.models.fusion import DualViewFusion
from src.preprocessing.video_loader import load_video
from src.postprocessing.output_formatter import logits_to_wells_typed
from train import build_leak_free_split, PipetteWellDataset

assert torch.cuda.is_available(), 'CUDA not available — switch the notebook accelerator to GPU'
assert torch.cuda.device_count() >= 1
print('Smoke test 1 PASS — imports succeed; CUDA device available.')

### Smoke test 2: leak-free split unit tests

Runs the 11 unit tests that prove `build_leak_free_split()` cannot leak
synthetic augmentations of val clips into the training set.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_leak_free_split.py', '-v'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Smoke test 2 FAIL — leak-free split tests did not all pass.')
print('Smoke test 2 PASS — leak-free split correctness verified.')

### Smoke test 3: split on the live dataset

Run `build_leak_free_split` on the actual real + synthetic labels and
verify that none of the four leakage conditions trigger.

In [ ]:
import json
real = json.load(open(DATA_DIR / 'labels.json'))
synth_path = DATA_DIR / 'labels_synthetic.json'
synth = json.load(open(synth_path)) if synth_path.exists() else []

if synth:
    train, val = build_leak_free_split(real, synth, val_split=0.2, seed=42)
    val_ids = {v['clip_id_FPV'] for v in val}
    train_synth_sources = {t['source_fpv'] for t in train if t.get('synthetic')}
    assert not (train_synth_sources & val_ids), 'LEAK: train-synth derived from val'
    assert not any(v.get('synthetic') for v in val), 'LEAK: synthetic in val'
    print(f'Live split: real={len(real)}  val_real={len(val)}  '
          f'train_real={sum(1 for t in train if not t.get("synthetic"))}  '
          f'train_synth={sum(1 for t in train if t.get("synthetic"))}')
    print('Smoke test 3 PASS — live leak-free split verified on actual data.')
else:
    print('Smoke test 3 SKIP — no labels_synthetic.json (real-only mode).')

### Smoke test 4: model forward pass on CUDA

Build a `DualViewFusion` and push a small dummy batch through it on the
GPU. Catches CUDA OOM, missing weights, dtype mismatches, and any
regression in the model definition before they cost a real epoch.

In [ ]:
import torch
from src.models.fusion import DualViewFusion

device = torch.device('cuda')
model = DualViewFusion(
    num_rows=8, num_columns=12, shared_backbone=True,
    use_lora=True, lora_rank=4, temporal_layers=1, img_size=518,
).to(device)
model.eval()

B, T, C, H, W = 2, 8, 3, 518, 518
fpv = torch.randn(B, T, C, H, W, device=device)
topview = torch.randn(B, T, C, H, W, device=device)

with torch.no_grad():
    row_logits, col_logits, type_logits = model(fpv, topview)

assert row_logits.shape == (B, 8), row_logits.shape
assert col_logits.shape == (B, 12), col_logits.shape
assert type_logits.shape == (B, 3), type_logits.shape
print(f'Forward pass shapes OK: row {tuple(row_logits.shape)} '
      f'col {tuple(col_logits.shape)} type {tuple(type_logits.shape)}')
print(f'Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')
print('Smoke test 4 PASS — model forward on CUDA works.')

del model, fpv, topview, row_logits, col_logits, type_logits
torch.cuda.empty_cache()

### Smoke test 5: dataset loads a real sample

In [ ]:
import json
from train import PipetteWellDataset

real = json.load(open(DATA_DIR / 'labels.json'))
ds = PipetteWellDataset(
    str(DATA_DIR), real[:2],
    num_frames=8, img_size=518, augment=False,
)
fpv, top, row_lbl, col_lbl = ds[0]
assert fpv.shape == (8, 3, 518, 518), fpv.shape
assert top.shape == (8, 3, 518, 518), top.shape
assert row_lbl.shape == (8,)
assert col_lbl.shape == (12,)
print(f'Loaded sample: fpv {tuple(fpv.shape)} top {tuple(top.shape)} '
      f'row_active={int(row_lbl.sum())} col_active={int(col_lbl.sum())}')
print('Smoke test 5 PASS — dataset can read a clip.')

### Smoke test 6: resume checkpoint loads (if attached)

In [ ]:
if RESUME_FROM is not None:
    ckpt = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    cfg = ckpt.get('model_config', {})
    test_model = DualViewFusion(
        num_rows=cfg.get('num_rows', 8),
        num_columns=cfg.get('num_columns', 12),
        shared_backbone=cfg.get('shared_backbone', True),
        use_lora=cfg.get('use_lora', True),
        lora_rank=cfg.get('lora_rank', 4),
        temporal_layers=cfg.get('temporal_layers', 1),
        img_size=cfg.get('img_size', 448),
    )
    test_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint loaded: epoch={ckpt["epoch"]+1} jaccard={ckpt["jaccard"]:.4f} '
          f'val_loss={ckpt["val_loss"]:.4f}')
    print(f'Model config: {cfg}')
    print('Smoke test 6 PASS — resume checkpoint loads cleanly.')
    del test_model, ckpt
else:
    print('Smoke test 6 SKIP — no checkpoint attached (fresh run from DINOv2 pretrained).')

## 6. Launch training

All smoke tests passed → safe to burn GPU hours.

Strategy:
- Stage data into a writable directory (Kaggle's `/kaggle/input/` is
  read-only; some training-time logic writes alongside the labels file).
  We use symlinks for the .mp4 files to avoid a 1+ GB copy.
- Save logs and the best checkpoint to `/kaggle/working/`, which Kaggle
  persists as notebook output.
- Keep the 12-hour Kaggle session limit in mind: if you hit it, the
  saved `best.pt` becomes the resume input for the next session.

In [ ]:
import os
from pathlib import Path

WORKING_DATA = Path('/kaggle/working/data/pipette_well_dataset')
WORKING_DATA.mkdir(parents=True, exist_ok=True)

# Symlink everything from the read-only dataset (fast — no copy).
for p in DATA_DIR.iterdir():
    dst = WORKING_DATA / p.name
    if not dst.exists():
        dst.symlink_to(p)

print(f'Staged data dir: {WORKING_DATA}')
print(f'  files: {len(list(WORKING_DATA.iterdir()))}')

In [ ]:
import os, subprocess

env = os.environ.copy()
env['DATA_DIR']      = str(WORKING_DATA)
env['LABELS']        = str(WORKING_DATA / 'labels.json')
env['OUTPUT_DIR']    = '/kaggle/working/checkpoints'
env['TRAINING_OUTPUT_DIR'] = '/kaggle/working/training_results'
env['DEVICE']        = 'cuda'
env['EPOCHS']        = str(EPOCHS)
env['BATCH_SIZE']    = str(BATCH_SIZE)
env['USE_SYNTHETIC'] = str(USE_SYNTHETIC)
env['WEIGHT_DECAY']  = str(WEIGHT_DECAY)
env['LORA_RANK']     = str(LORA_RANK)
env['DROPOUT']       = str(DROPOUT)
env['NUM_FOLDS']     = str(NUM_FOLDS)
env['FOLD_INDEX']    = str(FOLD_INDEX)
env['RESUME']        = str(RESUME_FROM) if RESUME_FROM else ''
if TRAINING_VERS:
    env['TRAINING_VERS'] = TRAINING_VERS

Path(env['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
Path(env['TRAINING_OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)

print('Launching run_training.sh -- output streams below.')
print('-' * 60)
subprocess.run(['bash', 'run_training.sh'], env=env, check=False)

## 7. Persist outputs

Kaggle preserves `/kaggle/working/` as the notebook's *output* (capped
at 20 GB). The training script already writes to that path, so the
checkpoint and log are persisted automatically.

**To resume in a future session:**
1. Open the completed run, click *Save Version → Quick Save* if you
   haven't already.
2. From the *Output* tab, *Save as Dataset* on `/kaggle/working/checkpoints/best.pt`.
3. Use that dataset slug for `CHECKPOINT_DATASET_SLUG` in the next run.

In [ ]:
from pathlib import Path

ckpt = Path('/kaggle/working/checkpoints/best.pt')
if ckpt.exists():
    print(f'best.pt size: {ckpt.stat().st_size / 1e6:.1f} MB')
else:
    print('No best.pt produced — check the training log above for errors.')

logs = sorted(Path('/kaggle/working/training_results').glob('training_*.log'))
if logs:
    print(f'Log: {logs[-1]} ({logs[-1].stat().st_size / 1024:.0f} KB)')